ODE:
$$\frac{dy(t)}{dt} + 0.05 y(t) = 0.2F(t), \qquad y(0) = 3.1$$
Where $F(t)$ is Real GDP data and $y(t)$ represents capital stock.

## Imports

In [ ]:
# Cell 1 - installs + clone
!pip install --quiet torch torchvision jax jaxlib keras_sig numpy matplotlib scipy pytorch-minimize
!git clone https://github.com/CharliePyle4/BSK_ODE.git /content/BSK_ODE 2>/dev/null || git -C /content/BSK_ODE pull

# Cell 2 - import
import sys
sys.path.insert(0, '/content/BSK_ODE')
from bsk_ode.sollow import *
print("Using device:", device, "torch", torch.__version__)

In [ ]:


# Load Data

df = pd.read_csv("econ_f_t_gdp_normalized_time.csv")
df["t"] = pd.to_datetime(df["t"])

t_forcing = df["t_norm"]
F_vals_original = 0.2 * df["f"]
dt_original = t_forcing[1] - t_forcing[0]
F_star_original = trapezoidal_array(F_vals_original, dt_original)

sol = pd.read_csv("sollow_ode_solution.csv")
t_true = sol["t"].values

forcing_data = pd.read_csv("integrated_f.csv")
t_vals_f = forcing_data["t_norm"].to_numpy(dtype=np.float64)
f_vals_f = forcing_data["f_integrated"].to_numpy(dtype=np.float64)

t_vals_f_tensor = torch.tensor(t_vals_f, dtype=torch.float64)
f_vals_f_tensor = torch.tensor(f_vals_f, dtype=torch.float64)

fullpath = torch.stack((t_vals_f_tensor, f_vals_f_tensor), dim=1)

num_partitions = len(fullpath)
total_points = len(fullpath)
partition_size = total_points // num_partitions

t_vals = t_vals_f_tensor
dt = float((t_vals[1] - t_vals[0]).item())
N = len(t_vals)


# Path Construction 
paths_nb = []  
for i in range(num_partitions):
    end_index = (i + 1) * partition_size
    if i == num_partitions - 1:
        end_index = total_points

    path_nb = []
    for j in range(end_index):
        t_val = forcing_data.iloc[j, 0]
        f_val = forcing_data.iloc[j, 1]
        ta1 = np.power(t_val, 1)
        path_nb.append([ta1, f_val])

    paths_nb.append(path_nb)


# Parameters

signature_level = 3
lambda_econ = 0.05
n0 = 50
retrain_every = 10


# Full Batch 

path_tensor_nb = torch.tensor(np.array(paths_nb[-1]), dtype=torch.float64).unsqueeze(0)
sigs_nb = signatory.signature(
    path_tensor_nb, stream=True, depth=signature_level,
    basepoint=path_tensor_nb[:, 0, :], scalar_term=True
).squeeze(0).detach().numpy()
S_nb = sigs_nb
N_full = len(S_nb)

K_nb = np.zeros((N_full, N_full), dtype=np.float64)
K1_nb = np.zeros((N_full, N_full), dtype=np.float64)
for i in range(N_full):
    for j in range(N_full):
        K_nb[i, j] = np.dot(S_nb[i], S_nb[j])
for j in range(N_full):
    col = K_nb[:, j]
    integrated = trapezoidal_array(col, dt)
    for i in range(N_full):
        K1_nb[i, j] = integrated[i]

integrated_f = pd.read_csv("integrated_f.csv")
F_star = integrated_f["f_integrated"].to_numpy(dtype=np.float64)

Psi_nb = K_nb + lambda_econ * K1_nb
A_nb = np.linalg.lstsq(Psi_nb, F_star, rcond=None)[0]
F_hat_nb = Psi_nb @ A_nb
K_A_nb   = K_nb @ A_nb

# Solow Solution

df_solow = pd.read_csv("sollow_ode_solution.csv")
time_col = df_solow.columns[0]
y_col    = df_solow.columns[-1]
t_solow  = np.asarray(df_solow[time_col], dtype=float)
y_solow  = np.asarray(df_solow[y_col],   dtype=float)

time_plot = t_vals_f
y_solow_interp_full = np.interp(time_plot, t_solow, y_solow)


# Plot 1 — 1x2: Forcing Fit & Solution Reconstruction

set_professional_plot_style()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Forcing fit
axes[0].plot(time_plot, F_star,   color="black", linewidth=1.5, label="Reference f(t)")
axes[0].plot(time_plot, F_hat_nb, color="blue",   linestyle="--", linewidth=1.5, label="Calibrated f(t)")
axes[0].set_title("Integrated Forcing Fit")
axes[0].set_xlabel("Normalized time t")
axes[0].set_ylabel("f(t)")
format_axes(axes[0])

# Right: Solution reconstruction
axes[1].plot(time_plot, y_solow_interp_full, color="black", linewidth=1.5, label="Reference y(t)")
axes[1].plot(time_plot, K_A_nb,              color="blue",   linestyle="--", linewidth=1.5, label="Reconstructed y")
axes[1].set_title("Solution Reconstruction")
axes[1].set_xlabel("Normalized time t")
axes[1].set_ylabel("y(t)")
format_axes(axes[1])

fig.tight_layout()
plt.show()

#Relative MSE — Forcing and Solution



# Forcing relative MSE
rel_mse_F_nb = rel_mse_np(F_hat_nb, F_star)

# Solution relative MSE
rel_mse_y_nb = rel_mse_np(K_A_nb, y_solow_interp_full)


print("\n" + "=" * 43)
print("Full-Batch Relative MSE Summary")
print("=" * 43)
print(f"{'Quantity':25s} {'Non-Branched':>15s}")
print("-" * 43)

print(
    f"{'Forcing f(t)':25s} "
    f"{rel_mse_F_nb:>15.6e} "
)

print(
    f"{'Solution y(t)':25s} "
    f"{rel_mse_y_nb:>15.6e} "
)

print("=" * 43)

In [ ]:
# Rolling Retrain Setup — non-branched only

F_star_torch  = torch.tensor(F_star,  dtype=torch.float64)
t_solow_torch = torch.tensor(t_solow, dtype=torch.float64)
y_solow_torch = torch.tensor(y_solow, dtype=torch.float64)

if len(y_solow_torch) != len(t_vals) or not torch.allclose(t_solow_torch, t_vals):
    y_true_interp = interpolate_to_grid(t_solow_torch, y_solow_torch, t_vals)
else:
    y_true_interp = y_solow_torch.clone()



state_nb = build_state_nonbranched(
    paths_nb=paths_nb,
    n0=n0,
    signature_level=signature_level,
    lambda_econ=lambda_econ,
    dt=dt,
    t_vals=t_vals,
    F_star_torch=F_star_torch,
    y_true_interp=y_true_interp
)

print("NON-BRANCHED TRAINING DONE.")




print("Running non-branched rolling prediction...")
res_nb = rolling_online_predict_econ_nonbranched(
    state_nb,
    retrain_every=retrain_every
)

print("NON-BRANCHED PREDICTION DONE.")




idx_train = torch.arange(0, n0 + 1)
idx_test = torch.arange(n0 + 1, res_nb["end_idx"] + 1)
idx_full = torch.arange(0, res_nb["end_idx"] + 1)

# Training errors
mse_train_F = mse_torch(res_nb["F_pred"][idx_train], F_star_torch[idx_train])
rel_mse_train_F = rel_mse_torch(res_nb["F_pred"][idx_train], F_star_torch[idx_train])

mse_train_y = mse_torch(res_nb["y_pred"][idx_train], y_true_interp[idx_train])
rel_mse_train_y = rel_mse_torch(res_nb["y_pred"][idx_train], y_true_interp[idx_train])

# Testing errors
mse_test_F = mse_torch(res_nb["F_pred"][idx_test], F_star_torch[idx_test])
rel_mse_test_F = rel_mse_torch(res_nb["F_pred"][idx_test], F_star_torch[idx_test])

mse_test_y = mse_torch(res_nb["y_pred"][idx_test], y_true_interp[idx_test])
rel_mse_test_y = rel_mse_torch(res_nb["y_pred"][idx_test], y_true_interp[idx_test])

# Full prediction errors
mse_full_F = mse_torch(res_nb["F_pred"][idx_full], F_star_torch[idx_full])
rel_mse_full_F = rel_mse_torch(res_nb["F_pred"][idx_full], F_star_torch[idx_full])

mse_full_y = mse_torch(res_nb["y_pred"][idx_full], y_true_interp[idx_full])
rel_mse_full_y = rel_mse_torch(res_nb["y_pred"][idx_full], y_true_interp[idx_full])


print("\n" + "=" * 86)
print("Non-Branched Error Summary")
print("=" * 86)
print(f"{'Quantity':25s} {'MSE':>18s} {'Relative MSE':>18s} {'Relative MSE (%)':>20s}")
print("-" * 86)

print(f"{'Training forcing':25s} {mse_train_F:>18.6e} {rel_mse_train_F:>18.6e} {100 * rel_mse_train_F:>19.4f}%")
print(f"{'Training solution':25s} {mse_train_y:>18.6e} {rel_mse_train_y:>18.6e} {100 * rel_mse_train_y:>19.4f}%")

print(f"{'Testing forcing':25s} {mse_test_F:>18.6e} {rel_mse_test_F:>18.6e} {100 * rel_mse_test_F:>19.4f}%")
print(f"{'Testing solution':25s} {mse_test_y:>18.6e} {rel_mse_test_y:>18.6e} {100 * rel_mse_test_y:>19.4f}%")

print(f"{'Full forcing':25s} {mse_full_F:>18.6e} {rel_mse_full_F:>18.6e} {100 * rel_mse_full_F:>19.4f}%")
print(f"{'Full solution':25s} {mse_full_y:>18.6e} {rel_mse_full_y:>18.6e} {100 * rel_mse_full_y:>19.4f}%")

print("=" * 86)


set_professional_plot_style()

t_train = t_vals[idx_train]
t_test = t_vals[idx_test]
t_split = float(t_vals[n0].item())

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharex=True)



ax = axes[0]

# True forcing
ax.plot(
    t_train.tolist(),
    F_star_torch[idx_train].tolist(),
    color="black",
    linewidth=1.6,
    label="True f(t)"
)
ax.plot(
    t_test.tolist(),
    F_star_torch[idx_test].tolist(),
    color="black",
    linewidth=1.6
)

# Predicted forcing
ax.plot(
    t_train.tolist(),
    res_nb["F_pred"][idx_train].tolist(),
    color="red",
    linestyle="--",
    linewidth=1.6,
    label="Predicted f(t)"
)
ax.plot(
    t_test.tolist(),
    res_nb["F_pred"][idx_test].tolist(),
    color="red",
    linestyle="--",
    linewidth=1.6
)

# Train/test split
ax.axvline(
    x=t_split,
    color="gray",
    linestyle=":",
    linewidth=1.5,
    label="Train/test split"
)

# Retrain markers
for r_idx in res_nb["retrain_indices"]:
    ax.axvline(
        x=float(t_vals[r_idx].item()),
        color="gray",
        linestyle=":",
        linewidth=0.7,
        alpha=0.25
    )

ax.set_title("Forcing Fit")
ax.set_xlabel("Normalized time t")
ax.set_ylabel("f(t)")
format_axes(ax)

ax = axes[1]

# True solution
ax.plot(
    t_train.tolist(),
    y_true_interp[idx_train].tolist(),
    color="black",
    linewidth=1.6,
    label="Reference y(t)"
)
ax.plot(
    t_test.tolist(),
    y_true_interp[idx_test].tolist(),
    color="black",
    linewidth=1.6
)

# Predicted solution
ax.plot(
    t_train.tolist(),
    res_nb["y_pred"][idx_train].tolist(),
    color="blue",
    linestyle="--",
    linewidth=1.6,
    label="Predicted y(t)"
)
ax.plot(
    t_test.tolist(),
    res_nb["y_pred"][idx_test].tolist(),
    color="blue",
    linestyle="--",
    linewidth=1.6
)

# Train/test split
ax.axvline(
    x=t_split,
    color="gray",
    linestyle=":",
    linewidth=1.5,
    label="Train/test split"
)

# Retrain markers
for r_idx in res_nb["retrain_indices"]:
    ax.axvline(
        x=float(t_vals[r_idx].item()),
        color="gray",
        linestyle=":",
        linewidth=0.7,
        alpha=0.25
    )

ax.set_title("Solution Fit")
ax.set_xlabel("Normalized time t")
ax.set_ylabel("y(t)")
format_axes(ax)


fig.tight_layout()
plt.show()

In [ ]:
# Relative MSE Summary — Non-Branched



idx_train = torch.arange(0, n0 + 1)
idx_test  = torch.arange(n0 + 1, res_nb["end_idx"] + 1)
idx_full  = torch.arange(0, res_nb["end_idx"] + 1)


# -------------------------
# Training relative MSEs
# -------------------------
rel_mse_train_F = rel_mse_torch(
    res_nb["F_pred"][idx_train],
    F_star_torch[idx_train]
)

rel_mse_train_y = rel_mse_torch(
    res_nb["y_pred"][idx_train],
    y_true_interp[idx_train]
)


# -------------------------
# Testing relative MSEs
# -------------------------
rel_mse_test_F = rel_mse_torch(
    res_nb["F_pred"][idx_test],
    F_star_torch[idx_test]
)

rel_mse_test_y = rel_mse_torch(
    res_nb["y_pred"][idx_test],
    y_true_interp[idx_test]
)


# -------------------------
# Full prediction relative MSEs
# -------------------------
rel_mse_full_F = rel_mse_torch(
    res_nb["F_pred"][idx_full],
    F_star_torch[idx_full]
)

rel_mse_full_y = rel_mse_torch(
    res_nb["y_pred"][idx_full],
    y_true_interp[idx_full]
)


# -------------------------
# Print table
# -------------------------
print("\n" + "=" * 72)
print("Non-Branched Relative MSE Summary")
print("=" * 72)
print(f"{'Quantity':25s} {'Relative MSE':>18s} {'Relative MSE (%)':>20s}")
print("-" * 72)

print(f"{'Training forcing':25s} {rel_mse_train_F:>18.6e} {100 * rel_mse_train_F:>19.4f}%")
print(f"{'Training solution':25s} {rel_mse_train_y:>18.6e} {100 * rel_mse_train_y:>19.4f}%")

print(f"{'Testing forcing':25s} {rel_mse_test_F:>18.6e} {100 * rel_mse_test_F:>19.4f}%")
print(f"{'Testing solution':25s} {rel_mse_test_y:>18.6e} {100 * rel_mse_test_y:>19.4f}%")

print(f"{'Full forcing':25s} {rel_mse_full_F:>18.6e} {100 * rel_mse_full_F:>19.4f}%")
print(f"{'Full solution':25s} {rel_mse_full_y:>18.6e} {100 * rel_mse_full_y:>19.4f}%")

print("=" * 72)